# DiT Next-Frame Prediction Evaluation

Loads a trained DiT checkpoint, runs `rollout()` on a DROID val sample with GT actions,
and visualizes predicted vs GT next frames.

Two visualization modes:
- **SH→RGB**: Direct DC-band SH extraction (`rgb = 0.5 + C0 * sh`)
- **Rendered**: Full Gaussian Splatting render via `diff-gaussian-rasterization`

In [1]:
import os, sys
sys.path.insert(0, os.path.expandvars("$GWM_PATH"))
import torch, numpy as np, matplotlib.pyplot as plt
from hydra import compose, initialize
from hydra.core.global_hydra import GlobalHydra

In [2]:
GWM_PATH = os.environ["GWM_PATH"]

GlobalHydra.instance().clear()
with initialize(config_path="../configs", version_base=None):
    cfg = compose(config_name="train_gwm", overrides=[
        "world_model.observation.use_gs=true",
        "world_model.reward.use_reward_model=false",
        "world_model.vae.use_vae=false",
        f"dataset.data_path={GWM_PATH}/data/",
    ])

print("Config loaded. context_length:", cfg.world_model.context_length)

Config loaded. context_length: 2


In [3]:
from gaussianwm.gwm_predictor import GaussianPredictor

CKPT = f"{GWM_PATH}/logs/gwm/checkpoints/model_latest.pt"
model = GaussianPredictor(cfg.world_model).cuda()
state_dict = torch.load(CKPT, map_location="cuda")
model.model.load_state_dict(state_dict)
model.eval()
print(f"Loaded DiT from {CKPT}")

/home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-20 13:49:46.282155: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-20 13:49:46.282181: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-20 13:49:46.282867: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-20 13:49:46.287095: I tensorflow/core/platform/cpu_feature_guard.cc:182] This

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]


/home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth


/home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(

🔥🔥🔥 Splatt3r Model loaded from /home/frankcholula/Workspace/gaussianwm/third_party/splatt3r/checkpoints/splatt3r_v1.0/epoch=19-step=1200.ckpt
[Model] Trainable parameters: 33.205088M
[Model] Total parameters: 33.598304M


/home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/frankcholula/Workspace/gaussianwm/gaussianwm/vq_model/lpips.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickli

loaded pretrained LPIPS loss from /home/frankcholula/Workspace/gaussianwm/ckpt/lpips/vgg.pth
Loaded DiT from /home/frankcholula/Workspace/gaussianwm/logs/gwm/checkpoints/model_latest.pt


In [4]:
from gaussianwm.processor.datasets import build_gaussian_splatting_reconstruction_dataset

val_dataset = build_gaussian_splatting_reconstruction_dataset("val", cfg.dataset)
obs, action, reward = next(iter(val_dataset))
# obs:    [T, H, W, 3]  uint8,  T=segment_length (10)
# action: [T, 10]       float
print(f"obs: {obs.shape} {obs.dtype}, action: {action.shape}")

2026-02-20 13:50:02.600049: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-20 13:50:02.602017: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-02-20 13:50:02.786856: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


Loading existing dataset statistics from /home/frankcholula/Workspace/gaussianwm/data/droid_100/1.0.0/dataset_statistics_a1db6f7c9761e0c1120da4648c2200f575f382544cdf98b0b8587752b44cacc0.json.


2026-02-20 13:50:03.964777: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# droid_100: ===============================================================1.000000 #
######################################################################################

Threads per Dataset: %s [48]
Reads per Dataset: %s [48]
Constructing datasets...


2026-02-20 13:50:04.494368: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


Applying frame transforms on dataset...


/home/frankcholula/Workspace/gaussianwm/gaussianwm/processor/datasets.py:249: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  left_frames = torch.from_numpy(left_frames)


obs: torch.Size([10, 128, 128, 3]) torch.uint8, action: torch.Size([24, 10])


In [5]:
HORIZON = 8  # predict 8 future frames

# Stack first 2 RGB frames as [1, 6, H, W] in [0, 255] float
# rollout() divides by 255 internally
context = obs[:2].permute(0, 3, 1, 2).float()      # [2, 3, H, W]
obs_input = context.reshape(1, -1, 128, 128).cuda() # [1, 6, 128, 128]

# GT actions for the predicted steps
gt_actions = action[2:2+HORIZON].float().cuda()     # [8, 10]

def gt_policy(obs_t, t):
    return gt_actions[t].unsqueeze(0)  # [1, 10]

with torch.no_grad():
    rollout_obs, rollout_actions, rollout_rewards = model.rollout(
        obs_input, policy=gt_policy, horizon=HORIZON
    )
# rollout_obs: [1, HORIZON+1, 28, 128, 128]
print(f"rollout_obs: {rollout_obs.shape}")
assert rollout_obs.shape == (1, HORIZON + 1, 28, 128, 128), \
    f"Unexpected shape: {rollout_obs.shape}"

rollout_obs: torch.Size([1, 9, 28, 128, 128])


In [ ]:
# Wire up Gaussian Splatting renderer from the already-loaded Splatt3r submodule.
# regressor.py already appended these paths when GaussianPredictor was initialised,
# so the imports below should resolve immediately (modules are already in sys.modules).
from src.pixelsplat_src.cuda_splatting import render_cuda
from utils.geometry import build_covariance
from utils.geometry import normalize_intrinsics as _norm_K

H, W = 128, 128

# Standard pinhole intrinsics in pixel space: focal = image_width → ≈53° FoV
# After normalize_intrinsics: [[1, 0, 0.5], [0, 1, 0.5], [0, 0, 1]]
_K_px  = torch.tensor([[[float(W), 0., W/2.],
                          [0., float(H), H/2.],
                          [0.,      0.,    1.]]],
                        dtype=torch.float32, device="cuda")
_K_norm = _norm_K(_K_px, (H, W))           # [1, 3, 3]

# Identity camera-to-world (c2w=I): camera at origin, looking along +Z
# Gaussian means are in view1's camera frame → identity c2w renders view1's image
_C2W  = torch.eye(4, dtype=torch.float32, device="cuda").unsqueeze(0)  # [1, 4, 4]
_near = torch.tensor([0.1],   dtype=torch.float32, device="cuda")
_far  = torch.tensor([100.0], dtype=torch.float32, device="cuda")
_bg   = torch.zeros(1, 3,    dtype=torch.float32, device="cuda")


def render_gaussians(gauss_14hw: torch.Tensor) -> np.ndarray:
    """Render a [14, H, W] CUDA tensor → [H, W, 3] numpy RGB in [0, 1].

    14-dim layout (must match Splatt3r / GWM convention):
      0:3  → XYZ means  (in view1 camera frame)
      3:6  → scales     (linear, post-activation)
      6:10 → rotations  (quaternion xyzw)
     10:13 → SH DC band (3 colour channels, 1 coefficient each)
     13:14 → opacity    (sigmoid-activated, clamped to [0, 1])
    """
    N = H * W
    g = gauss_14hw.reshape(14, N).T.contiguous()   # [N, 14]

    means  = g[:, 0:3]                             # [N, 3]
    scales = g[:, 3:6]                             # [N, 3]
    rots   = g[:, 6:10]                            # [N, 4]  xyzw
    sh_dc  = g[:, 10:13]                           # [N, 3]
    opc    = g[:, 13:14].clamp(0, 1)               # [N, 1]

    covs = build_covariance(scales, rots)          # [N, 3, 3]

    rendered = render_cuda(
        _C2W,                              # [1, 4, 4]
        _K_norm,                           # [1, 3, 3]
        _near,                             # [1]
        _far,                              # [1]
        (H, W),
        _bg,                               # [1, 3]
        means.unsqueeze(0),                # [1, N, 3]
        covs.unsqueeze(0),                 # [1, N, 3, 3]
        sh_dc.unsqueeze(-1).unsqueeze(0),  # [1, N, 3, 1]  (degree-0 SH)
        opc.squeeze(-1).unsqueeze(0),      # [1, N]
    )  # → [1, 3, H, W]

    # render_cuda sets requires_grad=True internally (mean_gradients for rasterizer)
    return rendered[0].permute(1, 2, 0).clamp(0, 1).detach().cpu().numpy()  # [H, W, 3]


# ── Baseline SH→RGB helpers (no rendering) ────────────────────────────────────
C0 = 0.28209479177387814

def gauss_to_rgb(gauss_14hw: torch.Tensor) -> np.ndarray:
    """[14, H, W] → [H, W, 3] numpy via direct DC-band SH→RGB (fast baseline)."""
    sh = gauss_14hw[10:13]  # [3, H, W]
    return (0.5 + C0 * sh).clamp(0, 1).permute(1, 2, 0).detach().cpu().numpy()

def gt_frame_rgb(obs_thwc: torch.Tensor, t: int) -> np.ndarray:
    """Raw uint8 GT frame at time t → [H, W, 3] numpy in [0, 1]."""
    return obs_thwc[t].float().numpy() / 255.


print("Renderer ready  (identity c2w, focal=image_width)")
print("SH→RGB helpers ready  (gauss_to_rgb, gt_frame_rgb)")

In [7]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

psnrs_sh,     ssims_sh     = [], []
psnrs_render, ssims_render = [], []

for t in range(HORIZON):
    gt_t   = t + 2          # GT frame index (skip 2 context frames)
    gt_img = gt_frame_rgb(obs, gt_t)                # [H, W, 3] float [0,1]

    pred_gauss = rollout_obs[0, t+1, 14:].clone()   # [14, H, W]

    # ── SH→RGB ────────────────────────────────────────────────────────────
    pred_sh = gauss_to_rgb(pred_gauss)
    p_sh = psnr(gt_img, pred_sh, data_range=1.0)
    s_sh = ssim(gt_img, pred_sh, data_range=1.0, channel_axis=2)
    psnrs_sh.append(p_sh);  ssims_sh.append(s_sh)

    # ── Gaussian Splatting render ──────────────────────────────────────────
    pred_rend = render_gaussians(pred_gauss)
    p_r = psnr(gt_img, pred_rend, data_range=1.0)
    s_r = ssim(gt_img, pred_rend, data_range=1.0, channel_axis=2)
    psnrs_render.append(p_r);  ssims_render.append(s_r)

    print(f"t={gt_t:2d}  SH→RGB: PSNR={p_sh:.2f} dB  SSIM={s_sh:.4f}"
          f"   |   Rendered: PSNR={p_r:.2f} dB  SSIM={s_r:.4f}")

print(f"\nMean SH→RGB  :  PSNR={np.mean(psnrs_sh):.2f} dB   SSIM={np.mean(ssims_sh):.4f}")
print(f"Mean Rendered:  PSNR={np.mean(psnrs_render):.2f} dB   SSIM={np.mean(ssims_render):.4f}")

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [ ]:
fig, axes = plt.subplots(3, HORIZON, figsize=(HORIZON * 2, 6))

for t in range(HORIZON):
    gt_t       = t + 2
    pred_gauss = rollout_obs[0, t+1, 14:].clone()   # [14, H, W]
    gt_img     = gt_frame_rgb(obs, gt_t)
    pred_sh    = gauss_to_rgb(pred_gauss)
    pred_rend  = render_gaussians(pred_gauss)

    # Row 0: Ground truth
    axes[0, t].imshow(gt_img)
    axes[0, t].set_title(f"GT t={gt_t}", fontsize=8)
    axes[0, t].axis("off")

    # Row 1: SH→RGB (direct extraction, no rendering)
    axes[1, t].imshow(pred_sh)
    axes[1, t].set_title(f"SH→RGB\n{psnrs_sh[t]:.1f}dB", fontsize=7)
    axes[1, t].axis("off")

    # Row 2: Full Gaussian Splatting render
    axes[2, t].imshow(pred_rend)
    axes[2, t].set_title(f"Rendered\n{psnrs_render[t]:.1f}dB", fontsize=7)
    axes[2, t].axis("off")

axes[0, 0].set_ylabel("GT",       fontsize=10)
axes[1, 0].set_ylabel("SH→RGB",   fontsize=10)
axes[2, 0].set_ylabel("Rendered",  fontsize=10)

plt.suptitle("GWM DiT Rollout: GT vs SH→RGB vs Gaussian Splatting Render", fontsize=10)
plt.tight_layout()
plt.savefig("eval_rollout_rendered.png", dpi=150)
plt.show()
print("Saved to eval_rollout_rendered.png")

In [ ]:
dim_groups = {
    "XYZ":       (0,  3),
    "Scales":    (3,  6),
    "Rotations": (6,  10),
    "SH/Color":  (10, 13),
    "Opacity":   (13, 14),
}

# Build GT Gaussian features for the 8 predicted frames
gt_rgb_tensor = (
    obs[2:2+HORIZON]              # [8, H, W, 3] uint8
    .permute(0, 3, 1, 2)          # [8, 3, H, W]
    .float()
    .unsqueeze(0)                 # [1, 8, 3, H, W]
    .cuda()
)

with torch.no_grad():
    gt_gaussians = model._process_obs(gt_rgb_tensor / 255.)  # [1, 8, 14, H, W]

pred_gaussians = rollout_obs[0, 1:, 14:]  # [8, 14, H, W]

print(f"{'Property':<12}  MSE")
print("-" * 25)
for name, (s, e) in dim_groups.items():
    mse = ((pred_gaussians[:, s:e] - gt_gaussians[0, :, s:e]) ** 2).mean().item()
    print(f"{name:<12}: {mse:.6f}")

Property      MSE
-------------------------
XYZ         : 0.175528
Scales      : 0.000135
Rotations   : 0.000686
SH/Color    : 0.786092
Opacity     : 0.001369
